# m22 lab · Electrogram annotation

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

A local activation time is an annotated time on a local electrogram minus a reference time on the same beat. You see what the reference adds to the error, compare annotation rules on modeled unipolar and bipolar signals, measure how noise moves an annotation, wrap reentrant activation into a window of interest, and separate a local deflection from a far-field one.

| Part | What you do | Equation |
| --- | --- | --- |
| A | The reference and its error | eq-m22-lat |
| B | Annotation rules | eq-m22-intrinsic, eq-m22-onset |
| C | Noise moves annotations | eq-m22-jitter |
| D | The window of interest | eq-m22-woi |
| E | Far-field signals | eq-m22-farfield |

Times in ms, lengths in mm, speeds in mm/ms (= m/s). The front model, conduction velocity and electrode height follow m03. Noise levels, reference jitter, cycle length and far-field shapes are illustrative. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import annotation as AN, mapping as MP

## Part A · The reference and its error

Points are often collected one beat at a time, each timed against a reference electrogram on that beat
(eq-m22-lat). If the reference wobbles from beat to beat, every point carries that error, but points recorded together
on one beat share it, so their differences do not. Fit a plane front (m30) to points from one beat of a 10-electrode
catheter, and to the same number of points spread over 10 beats.

In [ ]:
rng = np.random.default_rng(22)
c_true = 1.0                                              # m/s, as in m03
sa, sr = 0.5, 3.0                                         # annotation and reference SDs (ms), illustrative
def front(p): return p[:, 0] / c_true                     # front along x, time in ms
errs = {"one beat": [], "ten beats": []}
for _ in range(2000):
    p = rng.uniform(-10, 10, (10, 2))
    t = front(p) + rng.normal(0, sa, 10)
    one = t - rng.normal(0, sr)                           # one reference error for all 10 points
    ten = t - rng.normal(0, sr, 10)                       # a different beat (and reference error) per point
    for key, times in (("one beat", one), ("ten beats", ten)):
        v = MP.plane_fit_velocity(p, times)
        errs[key].append(np.linalg.norm(v) - c_true)
for key, e in errs.items():
    print(f"{key:9s}: speed error median {np.median(e):+.3f} m/s, IQR {np.subtract(*np.percentile(e, [75, 25])):.3f} m/s")
print("SD of a LAT difference:", AN.lat_difference_sd(sa, sr, True), "ms within a beat;",
      round(AN.lat_difference_sd(sa, sr, False), 2), "ms across beats")

In [ ]:
# Checks
iqr = {k: np.subtract(*np.percentile(e, [75, 25])) for k, e in errs.items()}
assert iqr["ten beats"] > 3 * iqr["one beat"]

## Part B · Annotation rules

In the m03 model the unipolar steepest downstroke falls exactly when the front passes the electrode, and the bipolar
peak exactly when it passes the bipole midpoint, provided $d\cos\theta < 2\sqrt3\,a$ (eq-m22-intrinsic). An onset rule
(the first time the bipolar signal reaches a fraction $f$ of its peak) fires earlier, by an amount that grows with the
electrode's height (eq-m22-onset). Height changes from point to point with contact, so onset LATs carry a varying
bias.

In [ ]:
t = np.arange(-30, 30, 0.005)
print(" h (mm)   peak (ms)   onset f=0.1 (ms)   small-spacing formula (ms)")
for h in (0.5, 1.0, 2.0, 3.0):
    a = 0.3 + h
    b = AN.bipolar_trace(t, 1.0, a, 2.0)
    print(f"{h:6.1f} {AN.peak_time(t, b):10.2f} {AN.onset_time(t, b, 0.1):14.2f} {-AN.onset_lead(a, 1.0, 0.1):20.2f}")
a = 0.5
for d in (1.0, 2.0, 4.0):
    b = AN.bipolar_trace(t, 1.0, a, d)
    print(f"a = {a} mm, spacing {d} mm (limit {2 * np.sqrt(3) * a:.2f} mm): bipolar peak at {AN.peak_time(t, b):+.2f} ms")
plt.plot(t, AN.bipolar_trace(t, 1.0, 0.5, 4.0)); plt.xlim(-6, 6); plt.xlabel("time (ms)"); plt.ylabel("bipolar (relative)")
plt.title("spacing beyond the limit: the peak splits"); plt.show()

In [ ]:
# Checks
assert abs(AN.peak_time(t, AN.bipolar_trace(t, 1.0, 1.3, 2.0))) < 1e-3
assert abs(AN.peak_time(t, AN.bipolar_trace(t, 1.0, 0.5, 4.0))) > 1.0

## Part C · Noise moves annotations

A threshold crossing moves by the noise divided by the slope where it crosses (eq-m22-jitter). The onset of a
bipolar signal sits on its foot, where the slope is modest. The peak sits where the slope is zero, so its error depends
on the curvature instead. The steepest downstroke uses a derivative, which amplifies fast noise. Measure all three with
illustrative noise that varies over about 0.1 ms, at 2% and 5% of the bipolar peak-to-peak.

In [ ]:
from scipy import ndimage
dt = 0.05
t = np.arange(-20, 20, dt)
b = AN.bipolar_trace(t, 1.0, 1.3, 2.0)
u = AN.unipolar_trace(t, 1.0, 1.3)
t_on = AN.onset_time(t, b, 0.1)
slope = np.interp(t_on, t, np.gradient(b, t))
sd = {}
for level in (0.02, 0.05):
    sn = level * (b.max() - b.min())
    res = {"onset": [], "peak": [], "steepest (unipolar)": []}
    for _ in range(1000):
        n = ndimage.gaussian_filter1d(rng.normal(0, 1, t.size), 2); n *= sn / n.std()
        res["onset"].append(AN.onset_time(t, b + n, 0.1))
        res["peak"].append(AN.peak_time(t, b + n))
        res["steepest (unipolar)"].append(AN.steepest_downstroke(t, u + n))
    sd[level] = {k: np.std(v) for k, v in res.items()}
    print(f"noise {100 * level:.0f}%: " + ", ".join(f"{k} SD {v:.3f} ms" for k, v in sd[level].items())
          + f"; onset predicted {AN.crossing_jitter(sn, slope):.3f} ms")

In [ ]:
# Checks
sn = 0.02 * (b.max() - b.min())
assert abs(sd[0.02]["onset"] - AN.crossing_jitter(sn, slope)) < 0.2 * AN.crossing_jitter(sn, slope)
assert sd[0.05]["steepest (unipolar)"] > sd[0.05]["onset"]

## Part D · The window of interest

In a reentry every site activates once per cycle, so its time after the reference is only defined up to whole cycles.
The window of interest picks one copy (eq-m22-woi): $w_0 + ((t - w_0) \bmod T)$, kept only if it falls inside the window.
The map then shows earliest meeting latest where the window wraps, and moving the window moves that line. A window
shorter than the cycle leaves part of the circuit unannotated. The 250-ms cycle and 100-unit circuit are
illustrative.

In [ ]:
T, L = 250.0, 100.0
p = np.linspace(0, L, 401)[:-1]
t_true = AN.ring_activation(p, L, T, t_start=-120.0)
for w0, W in ((-150, 250), (-60, 250), (-150, 200)):
    w = AN.window_lat(t_true, w0, W, T)
    jump = p[1:][np.abs(np.diff(w)) > T / 2]
    print(f"window {w0} to {w0 + W} ms: {100 * np.mean(np.isfinite(w)):.0f}% annotated, wrap at {jump.round(1).tolist()}")
    plt.plot(p, w, label=f"{w0} to {w0 + W} ms")
plt.xlabel("position around the circuit"); plt.ylabel("assigned LAT (ms)"); plt.legend(); plt.show()

In [ ]:
# Checks
w = AN.window_lat(t_true, -150, 200, T)
assert abs(np.mean(np.isfinite(w)) - 0.8) < 0.01

## Part E · Far-field signals

A far-field deflection comes from tissue far from the electrode, so it is broad. In the m03 model the largest slope
divided by the peak-to-peak amplitude is $c/a$ (eq-m22-farfield): sharp for near tissue, dull for far. When the
far-field shape is known up to a gain, least squares can subtract it, but its estimate absorbs any local signal that
overlaps it. Blanking around the far-field time instead loses any local activation inside the blanked span. Here the
local signal is small, as in low-voltage tissue, so the raw steepest downstroke falls on the far field.

In [ ]:
t = np.arange(-100, 150, 0.01)
far = AN.gaussian_wave(t, 40.0, 10.0)                     # a broad far-field deflection (illustrative 10-ms width)
for name, x in (("near (a = 1.3 mm)", AN.unipolar_trace(t, 1.0, 1.3)), ("far field", far)):
    print(f"{name:18s}: sharpness {AN.sharpness(t, x):.3f} per ms")
print("model c/a for a = 1.3 mm:", round(AN.sharpness_model(1.3, 1.0), 3), "per ms")
for t_loc in (-30.0, 30.0, 40.0):
    local = AN.unipolar_trace(t, 1.0, 1.3, t0=t_loc, amplitude=0.2)   # a small local signal (illustrative)
    x = local + 3.0 * far
    res_x, g = AN.remove_far_field(x, far)
    print(f"local at {t_loc:+5.0f} ms: gain {g:.3f} (true 3), annotation raw {AN.steepest_downstroke(t, x):+7.2f}, "
          f"after removal {AN.steepest_downstroke(t, res_x):+7.2f}, blanked {bool(AN.blanked(t_loc, 40.0, 30.0, 60.0))}")

In [ ]:
# Checks
x = AN.unipolar_trace(t, 1.0, 1.3, t0=-30.0, amplitude=0.2) + 3.0 * far
assert abs(AN.steepest_downstroke(t, x) - 40.0) < 5 and abs(AN.steepest_downstroke(t, AN.remove_far_field(x, far)[0]) + 30.0) < 0.05

## Where to go next

- Add a second, smaller local deflection (a double potential) and compare earliest, largest and latest annotations.
- Combine rules: accept a bipolar peak only near a unipolar steepest downstroke.
- m24 interpolates these LATs into maps; m26 propagates annotation error into map uncertainty.